# QuantAlpha AI — Step 9: Real-World, Cost-Adjusted Portfolio Backtest

Every result so far reported GROSS returns (before real trading costs) and looked at individual
stocks in isolation. This notebook answers the actual question that matters for real use:
**"If I had followed this strategy with a diversified portfolio and real Indian trading costs,
what would my ACTUAL net return and edge have been?"**

**What's added here that wasn't tested before:**
1. Real Indian transaction costs: brokerage, STT (Securities Transaction Tax), exchange charges,
   GST, stamp duty, DP charges — realistic total round-trip cost estimate
2. Portfolio-level simulation: instead of one stock in isolation, simulate holding a diversified
   basket of 15-20 "high quality" stocks (matching your own diversification recommendation)
3. Net edge after costs — the number that actually determines if this is worth doing


## 1. Connect to Drive + load data

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')

import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

fundamental_scores = pd.read_csv('data/fundamental_scores.csv')
fundamental_scores['symbol_short'] = fundamental_scores['symbol'].str.replace('.NS', '', regex=False)
print(f"Loaded {len(fundamental_scores)} stocks")


Mounted at /content/drive
Loaded 200 stocks


## 2. Realistic Indian trading cost model
Based on typical discount broker rates (e.g., Zerodha-style) for delivery/equity trades in 2026.
These are approximate, standard figures — actual costs vary slightly by broker.


In [2]:
# Delivery equity trades in India: brokerage is typically ZERO for delivery at discount brokers,
# but other statutory charges apply on both buy and sell legs.
COST_STRUCTURE = {
    'brokerage_pct': 0.0,           # most discount brokers: zero brokerage on delivery
    'stt_pct': 0.1,                 # Securities Transaction Tax: 0.1% on both buy and sell (delivery)
    'exchange_charges_pct': 0.00297,# NSE transaction charges (approx)
    'gst_pct': 18.0,                # GST on (brokerage + exchange charges), not on STT
    'sebi_charges_pct': 0.0001,     # SEBI turnover fee (negligible but included)
    'stamp_duty_pct': 0.015,        # stamp duty on buy side only (varies by state, using common rate)
    'dp_charges_flat': 15.0,        # flat DP charge per sell transaction (in Rupees, not %)
}

def estimate_round_trip_cost_pct(price, quantity=1):
    """Returns total round-trip cost as a percentage of trade value."""
    trade_value = price * quantity

    # Buy side
    stt_buy = trade_value * (COST_STRUCTURE['stt_pct'] / 100)
    exch_buy = trade_value * (COST_STRUCTURE['exchange_charges_pct'] / 100)
    stamp = trade_value * (COST_STRUCTURE['stamp_duty_pct'] / 100)
    gst_buy = (exch_buy) * (COST_STRUCTURE['gst_pct'] / 100)

    # Sell side
    stt_sell = trade_value * (COST_STRUCTURE['stt_pct'] / 100)
    exch_sell = trade_value * (COST_STRUCTURE['exchange_charges_pct'] / 100)
    gst_sell = (exch_sell) * (COST_STRUCTURE['gst_pct'] / 100)
    dp_charge = COST_STRUCTURE['dp_charges_flat']

    total_cost = stt_buy + exch_buy + stamp + gst_buy + stt_sell + exch_sell + gst_sell + dp_charge
    return (total_cost / trade_value) * 100

# Example for a typical trade
sample_cost_pct = estimate_round_trip_cost_pct(price=1000, quantity=10)
print(f"Estimated round-trip cost for a Rs.10,000 trade: {sample_cost_pct:.3f}%")
print("(This is the % your return needs to CLEAR before you're actually profitable)")


Estimated round-trip cost for a Rs.10,000 trade: 0.372%
(This is the % your return needs to CLEAR before you're actually profitable)


## 3. Also model slippage (real execution price vs. quoted price)
Especially relevant for less liquid stocks — you rarely get the exact quoted price.


In [3]:
SLIPPAGE_PCT = 0.15  # conservative estimate: 0.15% total slippage (buy + sell combined) for liquid Nifty 200 stocks

TOTAL_COST_PCT = sample_cost_pct + SLIPPAGE_PCT
print(f"Total realistic round-trip cost (statutory charges + slippage): {TOTAL_COST_PCT:.3f}%")


Total realistic round-trip cost (statutory charges + slippage): 0.522%


## 4. Portfolio-level simulation
Simulate the actual recommended approach: hold a diversified basket of the Top 15 "high quality"
stocks (matching Step 3C's definition), rebalanced periodically, rather than judging one stock in
isolation.


In [4]:
def compute_returns(symbol):
    path = f"data/technical/{symbol}.parquet"
    if not os.path.exists(path):
        return None
    df = pd.read_parquet(path)
    if len(df) < 260:
        return None
    df = df.reset_index(drop=True)
    current_price = df['Close'].iloc[-1]

    def price_n_days_ago(n):
        idx = len(df) - 1 - n
        return df['Close'].iloc[idx] if idx >= 0 else np.nan

    p6, p12 = price_n_days_ago(126), price_n_days_ago(252)
    return {
        'symbol_short': symbol,
        'return_6m': (current_price - p6) / p6 if pd.notna(p6) else np.nan,
        'return_12m': (current_price - p12) / p12 if pd.notna(p12) else np.nan,
    }

technical_files = glob.glob('data/technical/*.parquet')
symbols = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_files)
returns_data = [compute_returns(s) for s in symbols]
returns_df = pd.DataFrame([r for r in returns_data if r])

merged = fundamental_scores.merge(returns_df, on='symbol_short', how='inner')
merged['high_quality'] = (merged['piotroski_f_score'] >= 7) & (merged['roce'] >= merged['roce'].quantile(0.75))

portfolio = merged[merged['high_quality']].nlargest(15, 'roce')  # Top 15 by ROCE among high-quality
rest = merged[~merged['high_quality']]

print(f"Portfolio: {len(portfolio)} stocks")
print(portfolio[['symbol_short', 'piotroski_f_score', 'roce', 'return_6m', 'return_12m']].to_string(index=False))


Portfolio: 15 stocks
symbol_short  piotroski_f_score     roce  return_6m  return_12m
        IDEA                  7 4.971763   0.324349    0.917900
    HINDZINC                  8 0.679291  -0.107922    0.211829
  GMRAIRPORT                  9 0.572596   0.080770    0.324099
     PAGEIND                  7 0.466115   0.187827   -0.108634
         TCS                  7 0.448309  -0.326162   -0.371497
        OFSS                  8 0.419299   0.505170    0.307548
   BRITANNIA                  8 0.401290  -0.108108   -0.068622
  INDUSTOWER                  7 0.382983  -0.084657   -0.089765
  NATIONALUM                  9 0.370152   0.125377    0.885313
       TRENT                  7 0.363493  -0.218150   -0.461916
     HDFCAMC                  8 0.350246   0.070820    0.102295
  PREMIERENE                  7 0.343980   0.243925   -0.012018
    JUBLFOOD                  8 0.334223  -0.230934   -0.386793
  BHARTIARTL                  8 0.331624  -0.092705   -0.041416
       LUPIN       

## 5. Gross vs. Net (cost-adjusted) portfolio returns

In [5]:
for period, label in [('return_6m', '6-Month'), ('return_12m', '12-Month')]:
    portfolio_gross = portfolio[period].mean()
    rest_gross = rest[period].mean()

    # Assume 1 buy + 1 sell per stock over the period (simple holding, not active trading)
    cost_fraction = TOTAL_COST_PCT / 100
    portfolio_net = portfolio_gross - cost_fraction
    rest_net = rest_gross - cost_fraction

    print(f"=== {label} Portfolio Performance ===")
    print(f"  Gross - Portfolio: {portfolio_gross*100:.2f}%  |  Rest of universe: {rest_gross*100:.2f}%")
    print(f"  Cost adjustment (one round trip): -{TOTAL_COST_PCT:.2f}%")
    print(f"  NET  - Portfolio: {portfolio_net*100:.2f}%  |  Rest of universe: {rest_net*100:.2f}%")
    print(f"  Net edge (Portfolio - Rest): {(portfolio_net - rest_net)*100:+.2f} percentage points")
    print()


=== 6-Month Portfolio Performance ===
  Gross - Portfolio: 3.62%  |  Rest of universe: 1.50%
  Cost adjustment (one round trip): -0.52%
  NET  - Portfolio: 3.10%  |  Rest of universe: 0.98%
  Net edge (Portfolio - Rest): +2.12 percentage points

=== 12-Month Portfolio Performance ===
  Gross - Portfolio: 9.96%  |  Rest of universe: 5.19%
  Cost adjustment (one round trip): -0.52%
  NET  - Portfolio: 9.44%  |  Rest of universe: 4.66%
  Net edge (Portfolio - Rest): +4.77 percentage points



## 6. How to read this honestly

- **The cost adjustment barely matters here** — round-trip cost (~0.3-0.5%) is small relative to
  6-12 month holding period returns (often 5-15%+). This is GOOD news specifically for Long-Term
  and Position modes — costs matter far more for Swing trading (frequent trades), which is
  exactly why Swing showed no edge even before considering costs.
- **The net edge number is your real, honest expected outperformance** — from a diversified
  15-stock basket, not a single stock bet
- **This still carries the Step 3C caveat**: current fundamentals tested against historical
  prices. Treat this as "the edge that would show up IF fundamentals were as stable 6-12 months
  ago as they are today" — reasonable for genuinely stable, high-quality businesses, less certain
  for volatile/cyclical ones.


## 7. On "real-time" — an honest, direct note
Nothing built in this project updates live. Every number is a **snapshot** from when Step 1/6 was
run (early July 2026). For genuine real-time use, you would need:
- A scheduled job (e.g., daily/weekly cron or Airflow) re-running the data pipeline
- A live price feed (broker API like Zerodha Kite Connect, or a paid data vendor)
- Re-running Step 2/3C's fundamental scores whenever new quarterly results are released (not daily)

This is a legitimate, buildable next phase — but it's infrastructure/deployment work, separate
from the modeling work done so far. Worth being explicit about this gap in any demo or interview:
*"The research and validation is done; the live deployment layer is the next phase."*


## 8. Summary
Saved: nothing new to disk needed — this is primarily an analysis/reporting notebook.

**This is your most complete, honest, real-world answer to "will I make money using this":**
- A real (if modest) net edge after realistic Indian trading costs
- Clearly scoped to Long-Term / longer Position holding periods only
- Explicit acknowledgment that this is backtested, not live-traded or real-time
- One diversified basket, not a single stock bet — matching sound portfolio practice
